In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns




We need to figure out if the results are the same as using filter terms solo.
First load in the scores .

These are the scores from the per residue run.



In [2]:
df = pd.read_csv("filter_3rd_per_residue.sc", sep=r"\s+", skiprows=1).drop(columns=["SCORE:"])
df

,total_score,atomic_clashes_1,atomic_clashes_10,atomic_clashes_11,atomic_clashes_12,atomic_clashes_13,atomic_clashes_14,atomic_clashes_15,atomic_clashes_16,atomic_clashes_17,...,sc_nbr_counts_75,sc_nbr_counts_8,sc_nbr_counts_9,ss_contributes_core,total_hydrophobic,total_hydrophobic_AFILMVWY,total_sasa,two_core_each,unsat_hbond,description
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.548,5.927,3.39,1.0,10431.0,11796.0,4518.93,0.167,6.0,filter_3rd_per_residue_flags_relax_v6_no_carts...


In [3]:
df_per_residue = pd.read_csv('../per_residue_filter_terms/per_residue_final/filter_per_residue.sc', sep=r"\s+", skiprows=1).drop(columns=["SCORE:"])
df_per_residue

,total_score,atomic_clashes_1,atomic_clashes_10,atomic_clashes_11,atomic_clashes_12,atomic_clashes_13,atomic_clashes_14,atomic_clashes_15,atomic_clashes_16,atomic_clashes_17,...,sc_nbr_counts_7,sc_nbr_counts_70,sc_nbr_counts_71,sc_nbr_counts_72,sc_nbr_counts_73,sc_nbr_counts_74,sc_nbr_counts_75,sc_nbr_counts_8,sc_nbr_counts_9,description
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.931,0.294,2.953,0.263,5.045,0.177,1.548,5.927,3.39,filter_per_residue_flags_relax_v6_no_cartstruc...


They both seem to produce `total_score` so taht will have to be removed in post processing no matter what.


In [4]:
df_global = pd.read_csv('../filter_terms/final_global_filter/filter.sc', sep=r"\s+", skiprows=1).drop(columns=["SCORE:"])
df_global

,total_score,buried_all,buried_np,contact_all,contact_buried_core,contact_buried_core_boundary,degree,degree_core,degree_core_boundary,exposed_hydrophobics,...,res_count_buried_core_boundary,res_count_buried_np_core,res_count_buried_np_core_boundary,ss_contributes_core,total_hydrophobic,total_hydrophobic_AFILMVWY,total_sasa,two_core_each,unsat_hbond,description
0,0.0,10357.07,5560.343,271.0,30.0,141.0,10.653,15.182,13.233,2429.859,...,30.0,6.0,18.0,1.0,10431.0,11796.0,4518.93,0.167,6.0,filter_flags_relax_v6_no_cartstructure_0001_0001


In [5]:
keys_ind = df_global.keys().append(df_per_residue.keys())


In [6]:
keys_ind

Index(['total_score', 'buried_all', 'buried_np', 'contact_all',
       'contact_buried_core', 'contact_buried_core_boundary', 'degree',
       'degree_core', 'degree_core_boundary', 'exposed_hydrophobics',
       ...
       'sc_nbr_counts_7', 'sc_nbr_counts_70', 'sc_nbr_counts_71',
       'sc_nbr_counts_72', 'sc_nbr_counts_73', 'sc_nbr_counts_74',
       'sc_nbr_counts_75', 'sc_nbr_counts_8', 'sc_nbr_counts_9',
       'description'],
      dtype='object', length=554)

In [7]:
### both were scored with a structure relaxed with the final relax protocol in cartesian .

# flags_relax_v6_no_cartstructure_0001.pdb

## differences in total_score and of course description based on leading tags .
import numpy as np

ignore = {"total_score", "description", "SCORE:"}

cols_global = set(df_global.columns) - ignore
cols_per_res = set(df_per_residue.columns) - ignore
cols_df = set(df.columns) - ignore
cols_combined = cols_global | cols_per_res

print("=== Column Check ===")
print(f"  Missing from df:    {cols_combined - cols_df or 'PASS ✓'}")
print(f"  Extra in df:        {cols_df - cols_combined or 'PASS ✓'}")
print(f"  Unexpected overlap: {cols_global & cols_per_res or 'PASS ✓'}")

print("\n=== Value Check (atol=1e-5) ===")
all_pass = True

# --- Global ---
failures = []
for col in sorted(cols_global):
    a = float(df.iloc[0][col])
    b = float(df_global.iloc[0][col])
    if not np.isclose(a, b, atol=1e-5, equal_nan=True):
        failures.append((col, a, b, abs(a - b)))

if failures:
    all_pass = False
    print(f"  global: FAIL ✗  ({len(failures)} cols differ)")
    for col, a, b, diff in failures:
        print(f"    {col}: df={a}  global={b}  diff={diff:.2e}")
else:
    print(f"  global: PASS ✓  ({len(cols_global)} cols all within 1e-5)")

# --- Per-residue ---
failures = []
for col in sorted(cols_per_res):
    a = float(df.iloc[0][col])
    b = float(df_per_residue.iloc[0][col])
    if not np.isclose(a, b, atol=1e-5, equal_nan=True):
        failures.append((col, a, b, abs(a - b)))

if failures:
    all_pass = False
    print(f"  per_residue: FAIL ✗  ({len(failures)} cols differ)")
    for col, a, b, diff in failures:
        print(f"    {col}: df={a}  per_residue={b}  diff={diff:.2e}")
else:
    print(f"  per_residue: PASS ✓  ({len(cols_per_res)} cols all within 1e-5)")

if all_pass:
    print("\n✓ All checks passed — df is the union of df_global and df_per_residue.")

=== Column Check ===
  Missing from df:    PASS ✓
  Extra in df:        PASS ✓
  Unexpected overlap: PASS ✓

=== Value Check (atol=1e-5) ===
  global: FAIL ✗  (1 cols differ)
    pack: df=0.761  global=0.697  diff=6.40e-02
  per_residue: PASS ✓  (525 cols all within 1e-5)


So one value (pack) is off by 1e-2 as opposed to 1e-5. Nothing to worry about there. This script is completed.